In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [7]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV

In [8]:
import pandas as pd
df = pd.read_csv('../output/Bach_chordify_roman_5.csv')
display(df.head())
df.drop(['file', 'position'], axis=1, inplace=True)

,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5,label,duration,file,position
0,0,0,0,0,0,16,0.25,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",0
1,0,0,0,0,16,528,0.25,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",1
2,0,0,0,16,528,656,0.25,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",2
3,0,0,16,528,656,128,0.50,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",3
4,0,16,528,656,128,130,0.50,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",4


In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Encode columns in for the use of roman chords.
lookback_cols = [c for c in df.columns if 'lookback' in c and df[c].dtype == 'object' or 'label' in c]

# Build the union of all chords for label encoding
all_chords = set(df['label'].unique())
for col in lookback_cols:
    all_chords |= set(df[col].dropna().unique())

le_chord = LabelEncoder()
le_chord.fit(sorted(all_chords))

# Overwrite each lookback column with the same mapping
for col in lookback_cols:
    df[col] = le_chord.transform(df[col])


display(df.head())

,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5,label,duration
0,0,0,0,0,0,13,0.25
1,0,0,0,0,16,226,0.25
2,0,0,0,16,528,291,0.25
3,0,0,16,528,656,74,0.50
4,0,16,528,656,128,76,0.50


In [10]:
from fractions import Fraction
from sklearn.preprocessing import LabelEncoder
X_chord = df.drop(columns=['label', 'duration'])
y_chord = df['label']

X_duration = df.drop(columns=['duration'])
# Convert float durations into Fraction objects
df['duration'] = df['duration'].apply(lambda x: Fraction(x).limit_denominator())
le_dur = LabelEncoder()
le_dur.fit(sorted(df['duration'].unique()))
# Encode fractional durations for classification
df['duration'] = le_dur.transform(df['duration'])
y_duration = df['duration']

print("Training samples:", X_chord.shape[0])
print("Feature dims:",    X_chord.shape[1])
display(X_chord.head())
display(y_duration.head())

Training samples: 200525
Feature dims: 5


,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5
0,0,0,0,0,0
1,0,0,0,0,16
2,0,0,0,16,528
3,0,0,16,528,656
4,0,16,528,656,128


0    2
1    2
2    2
3    5
4    5
Name: duration, dtype: int64

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import RBFSampler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
#Split training data, and take smaller subset of data overall for model fitting.
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_chord, y_chord, test_size=0.2, random_state=42)
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_duration, y_duration, test_size=0.2, random_state=42)

Xc_small, _, yc_small, _ = train_test_split(Xc_train, yc_train, train_size=0.05, random_state=42)
Xd_small, _, yd_small, _ = train_test_split(Xd_train, yd_train, train_size=0.05, random_state=42)
display(Xd_small.shape[0])

8021

In [33]:
svc_chord = SVC(gamma='auto', C=10, random_state=42, probability=True)
svc_duration = SVC(gamma='auto', C=10, random_state=42, probability=True)

In [17]:
from joblib import dump, load
from sklearn.metrics import accuracy_score

svc_chord.fit(Xc_small, yc_small)
svc_duration.fit(Xd_small, yd_small)

dump(svc_chord, 'svc_chord_roman5.joblib')
dump(svc_duration, 'svc_duration_roman5.joblib')

"""
svc_chord    = load('svc_chord_roman5.joblib')
svc_duration = load('svc_duration_roman5.joblib')
"""

y_pred_chord   = svc_chord.predict(Xc_test)
y_pred_duration = svc_duration.predict(Xd_test)

print("Test accuracy (chord):   ", accuracy_score(yc_test, y_pred_chord))
print("Test accuracy (duration):", accuracy_score(yd_test, y_pred_duration))

Test accuracy (chord):    0.00019947637451689316
Test accuracy (duration): 0.38616132651789054


In [ ]:
import numpy as np
import pandas as pd

chord_cols = clf_chord.feature_names_in_
dur_cols   = clf_duration.feature_names_in_

#Add starting point for the generation. For the sake of simplicity, starting point was set to beginning.
init_chord_ctx = list(Xc_test.iloc[0])
init_dur_ctx   = list(Xd_test.iloc[0])[: len(Xd_test.iloc[0]) - len(init_chord_ctx)]

steps      = 128
generated  = []
chord_ctx  = init_chord_ctx.copy()
dur_ctx    = init_dur_ctx.copy()

for step in range(steps):
    # Generate Chord
    Xc_df = pd.DataFrame([chord_ctx], columns=chord_cols)
    proba_c = clf_chord.predict_proba(Xc_df)[0]
    enc_chord = np.random.choice(clf_chord.classes_, p=proba_c)
    # Genderate Duration
    chord_ctx = chord_ctx[1:] + [enc_chord]
    Xd_df = pd.DataFrame([chord_ctx + dur_ctx], columns=dur_cols)
    proba_d = clf_duration.predict_proba(Xd_df)[0]
    next_dur = np.random.choice(clf_duration.classes_, p=proba_d)
    dur_ctx = dur_ctx[1:] + [next_dur]
    
    generated.append({
        'chord':    enc_chord,
        'duration': next_dur
    })

df_gen = pd.DataFrame(generated)
df_gen['chord'] = le_chord.inverse_transform(df_gen['chord'].values)
df_gen['duration'] = le_dur.inverse_transform(df_gen['duration'].values)
df_gen.to_csv('generated_sequence_batched.csv', index=False)
print(f"\nWrote {len(df_gen)} events to generated_sequence_full.csv")